# Probe re-fit and QC

Manual correction of the `brainreg_probe` probe fits. Full rationale, the angle
diagnosis and the per-subject outcomes are in `PROBE_REFIT.md`.

> **Kernel:** this notebook needs the **`histology`** conda env
> (`~/.conda/envs/histology`). The default `maze_ephys_si104` env has no
> `skimage` and cannot import the tracing module.

**What the tiers are**

| tier | `fit_method` | when |
|---|---|---|
| 1 | `override` | plane is right, placement is off |
| 2 | `trajectory_constrained` | plane is wrong — bound the tilt to the surgery, re-fit |
| 3 | `manual_track` | dye unusable — annotate entry + tip by hand. **Authoritative** |

**Who judges what.** The QC metrics catch an *algorithm* fitting the wrong thing
and apply only to tiers 1–2. Against a hand annotation they are descriptive:
`resid_*` and `signal_coverage` are measured against the Otsu-thresholded dye
mask, and a bad threshold is exactly why you would annotate by hand. A
`manual_track` fit is graded `annotated`, never `review`; only *you* set
`confidence`.

In [1]:
import sys
sys.path.insert(0, '.')          # run from code/histology_refit/
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import probe_refit as pr

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 60)
print('subjects:', pr.list_subjects())
print('surgical prior:', pr.SURGICAL_PRIOR)

subjects: ['ah08', 'ah09', 'ah10', 'ly05', 'ly06', 'ly07']
surgical prior: {'lateral_deg': 10.0, 'ap_deg': 0.0, 'tol_deg': 7.0}


## 1. Synthetic gate — run before trusting anything

Ground truth is a known (plane, params); dye is synthesised along that known
track, the fit is perturbed to reproduce each observed failure mode, and we check
both that QC flags it **and** that the truth is *not* flagged (otherwise the gate
is vacuous). It also asserts the AP-tilt/theta cancellation signature reproduces,
since that is what the ah10 diagnosis leans on.

In [ ]:
gate = pr.run_synthetic_controls()
assert gate.attrs['passed'], 'gate failed — do not trust anything below'

## 2. Where every subject stands

`lateral_deg` should be ≈10 (the nominal lateral approach), `ap_deg` ≈0, and
`theta_deg` ≈0 — a straight insertion has no reason to rotate the probe within
its own plane, so **a large `theta` is the "plane is wrong" alarm**.

In [ ]:
qc = pr.qc_table()
qc.drop(columns=['params_at_bounds'], errors='ignore')

## 3. Is the dye usable at all?

`extent_um` is how far the dye runs along the current trajectory — directly
comparable to `probe_depth`. A ratio far from 1 means the fit does not span its
own dye (ah10: 3654 µm of dye against a 2205 µm fit). `dist_to_centroid_um` says
whether the dye that was found is even the track (ly05: 3.8 mm away — it is not).

In [ ]:
diag = pd.concat([pr.signal_diagnostics(m, gammas=(1.5,)) for m in pr.list_subjects()],
                 ignore_index=True)
diag['dye_vs_fit'] = diag['extent_um'] / diag['fitted_depth_um']
diag

## 4. Compare strategies before choosing one

**Do not blanket-apply.** `fresh+depth` rescues ah10 and actively harms ah08,
whose dye is sparse (458 points over 58% of the probe) — re-deriving plane and
depth from a partial cloud truncates a fit that was already right. Each subject is
judged on its own evidence.

In [ ]:
STRATEGIES = {
    'baseline':    dict(optimize=False),
    'constrained': dict(optimize=True, trajectory_prior=pr.SURGICAL_PRIOR),
    'fresh_plane': dict(optimize=True, trajectory_prior=pr.SURGICAL_PRIOR, fresh_plane=True),
    'fresh+depth': dict(optimize=True, trajectory_prior=pr.SURGICAL_PRIOR,
                        fresh_plane=True, reset_depth=True),
}

def compare(subject, strategies=STRATEGIES):
    sig = pr.load_signal_df(subject)
    rows = []
    for name, kw in strategies.items():
        try:
            q = pr.refit_probe(subject, signal_df=sig, **kw)['qc']
            rows.append({'strategy': name, 'lat': q['lateral_deg'], 'ap': q['ap_deg'],
                         'theta': q['theta_deg'], 'depth': q['probe_depth'],
                         'resid_s2c': q['resid_signal2contact_um'],
                         'cov': q['signal_coverage'], 'tip': q['tip_structure'],
                         'ENTl_frac': q['target_fraction'], 'flags': ';'.join(q['flags'])})
        except Exception as exc:
            rows.append({'strategy': name, 'tip': f'ERROR {exc}'})
    return pd.DataFrame(rows)

compare('ah10')

## 5. ah10 — the correction

The automated plane was tilted **25.4° in AP** (the wrong plane for a lateral
approach) with a compensating **−24.3°** in-plane rotation, the two cancelling to
1.2°, and depth truncated to 2205 µm against 3654 µm of dye. The dye's own
principal axis is only ~7° from the surgical prior, so the fix is a fresh PCA
plane on the correctly-extracted cloud.

In the figure, dye is drawn **on top** of the contacts: where the fit is good the
red disappears under cyan.

In [ ]:
sig = pr.load_signal_df('ah10')
data = pr.load_volumes('ah10')
before = pr.refit_probe('ah10', signal_df=sig, optimize=False, data=data)
after  = pr.refit_probe('ah10', signal_df=sig, fresh_plane=True, reset_depth=True,
                        trajectory_prior=pr.SURGICAL_PRIOR, optimize=True, data=data)

pr.plot_fit_qc('ah10', before, data=data, signal_df=sig, title='ah10 — automated (before)')
pr.plot_fit_qc('ah10', after,  data=data, signal_df=sig, compare=before,
               title='ah10 — corrected (orange = automated)')
plt.show()
pd.DataFrame([before['qc'], after['qc']], index=['before', 'after'])[
    ['lateral_deg','ap_deg','theta_deg','probe_depth',
     'resid_signal2contact_um','signal_coverage','tip_structure','target_fraction']]

## 6. Tier 3 — manual annotation (ly05, and anywhere you disagree)

ly05's dye is unusable: 167 points spanning 97 µm whose centroid is **3.8 mm off
the track**, and every automated strategy fails or places contacts outside the
volume. This is the case the manual path exists for.

**Workflow.** Browse slices, read the entry point (brain surface) and tip
straight off the axes — they are voxel indices — then pass them to
`apply_manual_track`. Use `annotate_track` for an interactive slider, or
`slice_grid` for a static sheet. Both draw the **raw** volume with the threshold
mask only as an overlay, so faint dye Otsu discarded stays visible.

In [ ]:
# widen the range if the track is not in view; ticks are voxel indices
pr.slice_grid('ly05', axis='k', slices=np.arange(700, 860, 20), n=8)
plt.show()

In [ ]:
# interactive alternative (needs ipywidgets)
# pr.annotate_track('ly05')

In [ ]:
# --- fill these in from the slices, as (i, j, k) voxel coordinates ---
ENTRY = None    # e.g. (470, 150, 856)   brain surface
TIP   = None    # e.g. (474, 420, 873)   deepest point of the track

if ENTRY and TIP:
    res = pr.apply_manual_track('ly05', ENTRY, TIP)
    for note in pr.annotation_report('ly05', res)['notes']:
        print('•', note)
    pr.plot_fit_qc('ly05', res, title='ly05 — manual annotation')
    plt.show()
else:
    print('Set ENTRY and TIP above from the slice views.')

Saving an annotation. **You** set `confidence` — the harness never infers it.
`manual_track` + `confident` is the highest grade in the scheme, and the flag
travels with the data so downstream analyses can split or caveat on it.

In [ ]:
# pr.save_fit('ly05', res,
#             fit_method='manual_track',
#             confidence='confident',        # or 'uncertain'
#             note='Annotated from raw DiI; automated fit had no usable signal.',
#             manual_inputs=res['manual_inputs'])

## 7. ly07 — genuinely ambiguous, your call

All four strategies give a comparable residual (60–96 µm) and good coverage, so
the dye does not discriminate between them — yet the tip label swings between
**ENTl6a, ENTm5 and SUB**, and the recorded bank is dominated by SUB/ProS/ENTm
under every one. ly07 sits at the ENTl/ENTm/subiculum junction, where the
assignment is not resolvable at this method's precision.

Worth stating plainly: the fit objective is agreement with the **dye**.
`target_fraction` is a review flag and never an optimisation target — otherwise
one simply picks whichever strategy yields the most ENTl. A probe that genuinely
missed ENTl is a result, not a bug. Annotate it and mark `confidence` honestly.

In [ ]:
compare('ly07')

## 8. Final state and the downstream census

The recorded bank is the tip-most 705 µm — the only part that produces data — so
this table is what any structure-split analysis must be built on.

In [ ]:
final = pr.qc_table()
final.to_csv(pr.BRAINREG_DIR / 'probe_fit_qc.csv', index=False)
final.drop(columns=['params_at_bounds'], errors='ignore')

In [ ]:
# recorded-bank composition per subject, from the current (possibly corrected) fits
rows = []
for m in pr.list_subjects():
    fit = pr.load_fit(m)
    plane, params = pr.split_fit(fit)
    pdf = pr.project_probe(plane, params, pr.load_volumes(m))
    bank = pdf[pdf['probe_coords.y'] <= pr.RECORDED_BANK_MAX_UM]
    counts = bank['structure.acronym'].value_counts()
    rows.append({'subject': m, 'fit_method': fit.get('fit_method', 'auto'),
                 'confidence': fit.get('confidence', 'unset'),
                 **counts.head(6).to_dict()})
    pr.clear_volume_cache(m)
pd.DataFrame(rows).fillna(0)

<!-- phase2 -->
## 9. Per-shank **and** pooled anatomy

Both, always. They answer different questions:

* **Pooled** — what fraction of recording sites / units sit in a region. This sets
  the *n* for any region-split analysis.
* **Per-shank** — how anatomy varies across the probe's 750 µm span. Pooled numbers
  hide this: ly06's pooled 0.58 ENTl is really three shanks in LEC (0.54 / 0.77 /
  1.00) and the most posterior one entirely in **ENTm** — the probe straddles the
  LEC/MEC border. ly07's pooled 0.076 is really an anterior→posterior gradient.

> **Shank numbers carry a `?`.** Which physical end holds channels 0–95 cannot be
> recovered from the fit — see §10. The **AP ordering and the anatomy are
> orientation-independent and correct**; only the shank label is pending.

In [ ]:
res = {}
for m in pr.list_subjects():
    fit = pr.load_fit(m); plane, params = pr.split_fit(fit)
    res[m] = {'probe_df': pr.project_probe(plane, params, pr.load_volumes(m))}
    pr.clear_volume_cache(m)

for m, r in res.items():
    bank = r['probe_df'][r['probe_df']['probe_coords.y'] <= pr.RECORDED_BANK_MAX_UM]
    print(f"\n=== {m}   POOLED: "
          f"{bank['structure.acronym'].value_counts().head(4).to_dict()}")
    print(pr.shank_breakdown(r).to_string(index=False))

## 10. Why shank *numbers* are unverified — an exact degeneracy

Mirroring the probe (`u_axis → −u_axis`, `offset_x → −offset_x`, `theta → −theta`)
reproduces the contact positions to **0.0 µm** and the cost to **0.000%**. The four
shanks are geometrically identical and their centred x-set equals its own mirror,
so the two solutions are the same point cloud — **no dye-based method can tell them
apart**. Only the surgical record (which way the contact face pointed) can.

What it does *not* affect: the structure at a given physical position, verified
below. So the tables above are correct as printed.

In [ ]:
import numpy as np
m = 'ly07'
data = pr.load_volumes(m)
fit = pr.load_fit(m); plane, params = pr.split_fit(fit)
pm = dict(plane); pm['u_axis'] = -np.asarray(plane['u_axis'], float)
qm = dict(params); qm['offset_x'] = -params['offset_x']; qm['theta'] = -params['theta']

for lbl, (pl, pa) in [('original', (plane, params)), ('mirrored', (pm, qm))]:
    sb = pr.shank_breakdown({'probe_df': pr.project_probe(pl, pa, data)})
    print(f"{lbl:9s} by AP position ->", list(sb['top1']))
print('\nSHANK_ORDER_VERIFIED =', pr.SHANK_ORDER_VERIFIED)
pr.clear_volume_cache(m)

## 11. Depth is often *not identifiable* from dye

Dye is wiped onto the shank on the way in and commonly fades before the tip. When
it does, **no dye-based objective can know the probe went deeper** — measured
against a known truth, depth is under-read by up to 2.3 mm, and switching to
`signal2contact` barely helps. This is an information limit, not a cost-function
bug.

So `depth_over_dye` is reported with advisories `fit_stops_short_of_dye` (<0.9×)
and `depth_extrapolated_past_dye` (>1.3×) — both **advisory**, never hard flags. A
fit longer than its dye is expected; only an annotation or the surgical record can
settle depth when the dye is partial.

In [ ]:
pr.depth_objective_comparison(true_depth=4000.0)

### Depth sweep — the curve, not a verdict

For a subject whose depth is in question, sweep it and read the per-shank anatomy.
This is how ah10's depth was settled at 3774 µm (= its own dye extent), where all
four shanks sit entirely in ENTl; the automated fit had stopped at 0.81× that.
The layer progression along the track (ENTl6a → 5 → 3 → 2, then out of the brain
past ~4800 µm) is anatomically coherent and is a good sanity check.

In [ ]:
sw = pr.depth_sweep('ah10', depths=np.arange(2400, 5201, 400))
cols = ['depth_um', 'depth_over_dye'] + \
       [c for c in sw.columns if c.endswith('_top') or c.endswith('_ENTl')]
sw[cols]

<!-- placer -->
## 12. Interactive placement — put the shanks where your eye says

`probe_tool.place(subject)` opens sagittal + coronal DiI panels with sliders over the
shanks, initialised at that subject's **current saved fit** (verified to reproduce it to
0.000000 µm).

Controls are **entry + tip**, not angles. Two points fix the trajectory and depth
outright, which sidesteps the pivot problem — rotating about the surface anchor swung
ah10's tip 574 µm for a 9.7° change, so "what does an angle slider rotate about" has no
innocent answer. `theta`, width and shrinkage get their own sliders since two points
cannot fix them; `offset_x` is pinned to 0 because it is redundant with moving the entry.

Panels are **slab max-intensity projections** (default 400 µm), so all four shanks' dye
is visible at once — a single slice never contains them all. Atlas contours (ENTl / ENTm /
SUB) come from the **centre slice**, since a max-projection of label ids is meaningless.

A placement saved here is **authoritative**: `fit_method='manual_3d'`, graded `annotated`.
The harness reports metrics beside it but never re-optimises it against the dye or fails
it. Only mechanical validity (entry ≡ tip, coords outside the volume) blocks a save.

In [ ]:
import probe_tool as pt

tool = pt.place('ly05')      # or 'ly05' — the two still open
# move the sliders; the readout under the panels updates with
#   trajectory / dye fit / bank composition / per-shank ENTl

When you are happy, set **confidence** in the dropdown and press **SAVE placement**.
`confident` vs `uncertain` is your call — the harness never infers it — and the flag
travels with the data so downstream analyses can split or caveat on it.

To save from code instead of the button:

In [8]:
tool.save(confidence='confident',
          note='ah10 placed by hand off the DiI')


{
    'probe_depth': 3273.301086059759,
    'brain_shrinkage_pct': 0.0,
    'probe_width_scaling': 1.05,
    'theta': -0.16195496786464916,
    'offset_x': 0.0,
    'probe_name': 'ProbeA',
    'centroid': [646.0, 302.0, 883.5],
    'v_axis': [0.036660238959090125, -0.9898264518954334, -0.13747589609658797],
    'u_axis': [-0.9782016773282249, -0.06368130796069425, 0.1976516366961386],
    'normal': [0.20439546313876333, -0.1272331959219474, 0.9705844674750249],
    'surface_coord': [652.0, 140.0, 861.0],
    'fit_method': 'manual_3d',
    'confidence': 'confident',
    'note': 'ah10 placed by hand off the DiI',
    'shank_order_verified': False,
    'corrected_date': '2026-09-01T08:56:35',
    'manual_inputs': {
        'entry': [652.0, 140.0, 861.0],
        'tip': [640.0, 464.0, 906.0],
        'theta_deg': -9.27933612982127,
        'slab_um': 800.0,
        'placed_with': 'probe_tool.ProbePlacer'
    },
    'qc': {
        'lateral_deg': 7.907162702958458,
        'ap_deg': -2.100

### Redraw cost

Measured on ly07: **~280 ms** with the slab cache warm, **~400 ms** after a move that
forces a new projection, ~680 ms on the very first draw. Sliders use
`continuous_update=False` so a redraw happens on release, not during a drag.

This is dominated by matplotlib re-drawing ~2000 contacts, ~950 dye points and the
contours. The structure lookup that used to cost **4.4 s** is now **0.4 ms** — see §13.

## 13. Why the tool is possible at all: `fast_structure_labels`

`pit.get_structure_labels` loops in Python running a DataFrame `.query()` per contact —
**~4400 ms** for 2016 contacts, which would make any interactive tool impossible.
`probe_refit.fast_structure_labels` indexes the volume with numpy and maps only the
distinct ids present (~26), then expands back, so cost scales with unique structures
rather than contacts.

**Verified bit-identical on all six subjects** — acronym, name and id — at 4,500–10,000×.

One trap worth recording: the first version indexed a dense array *by structure id*.
Allen ids reach **614,454,277**, so that allocated three 614-million-entry object arrays
(**14.7 GB**) and got the process OOM-killed. The benchmark had missed it because the
allocation sat outside the timer. The dict + `np.unique` version below is both correct and
small.

In [ ]:
import time
data = pr.load_volumes('ah08')
fit = pr.load_fit('ah08'); plane, params = pr.split_fit(fit)
probe_df = pr.project_probe(plane, params, data)
coords = probe_df[[f'downsample_coords.{c}' for c in 'ijk']].values

from brainreg_probe import probeinterface_tracing as pit
t = time.time(); ref = pit.get_structure_labels(coords, data); t_ref = time.time()-t
t = time.time(); new = pr.fast_structure_labels(coords, data); t_new = time.time()-t
same = all(str(a) == str(b) for a, b in zip(new['acronym'], ref['acronym']))
print(f'identical: {same}   {t_ref*1000:.0f} ms -> {t_new*1000:.2f} ms '
      f'({t_ref/t_new:.0f}x)')
pr.clear_volume_cache('ah08')